# Sesion 8 — De Datos Crudos a Dataset Analizable
## Diplomado: Machine Learning en Seguros · FC UNAM
### 2 de mayo de 2026  ·  07:00 - 11:00 h  (4 horas)

---

> **Premisa de la sesion:** recibes los archivos crudos de la aseguradora.
> Tu trabajo: convertirlos en un dataset limpio, bien tipado y eficiente,
> resolviendo 7 problemas reales uno por uno.

---

**Prerequisito:** Debe existir la carpeta `datos/` con los archivos CSV.

## Las 7 Dudas que Resolvemos Hoy

| # | Duda | Herramienta |
|---|------|-------------|
| 1 | 46 columnas — ¿cuales necesito? | Taxonomia + `usecols` |
| 2 | Texto sucio: M/F/Masculino/Femenino | `str` operations |
| 3 | Fechas como texto: '15/04/2026' | `pd.to_datetime()` |
| 4 | API de reaseguro devuelve JSON | `read_json()` + `json_normalize()` |
| 5 | 90k filas, 11MB sin esperar | `chunks` + `category` |
| 6 | Downcast rompio precision de primas | Estrategia segura de `dtypes` |
| 7 | ¿Cuando cambiar a Polars? | `polars` — intro y comparativa |

---
## ACT 1 — El Dataset Crudo

### Duda 1: 46 columnas — ¿cuales necesito?

In [1]:
import pandas as pd
import numpy as np
import os, time

# ── Paso 1: Cargar TODO para entender que hay ────────────────────────────────
# Primera regla: antes de descartar, entende lo que tienes
df_todo = pd.read_csv('datos/cartera_polizas.csv', nrows=5)  # solo 5 filas para ver
print(f'Columnas totales: {len(df_todo.columns)}')
print()
print('Lista de columnas:')
for i, col in enumerate(df_todo.columns, 1):
    print(f'  {i:>2}. {col}')

Columnas totales: 46

Lista de columnas:
   1. id_poliza
   2. num_poliza
   3. id_contrato_interno
   4. folio_emision
   5. id_sistema_legacy
   6. nombre
   7. apellido_paterno
   8. apellido_materno
   9. nombre_completo
  10. rfc
  11. fecha_nacimiento
  12. edad
  13. sexo
  14. estado_civil
  15. ocupacion
  16. nivel_educacion
  17. ramo
  18. plan
  19. fecha_emision
  20. fecha_inicio_vigencia
  21. fecha_fin_vigencia
  22. num_renovaciones
  23. status_poliza
  24. motivo_baja
  25. canal_venta
  26. marca_vehiculo
  27. modelo_vehiculo
  28. tipo_vehiculo
  29. suma_asegurada
  30. deducible
  31. prima_neta
  32. prima_total
  33. cuota_prima
  34. forma_pago
  35. num_cuotas
  36. agente_id
  37. estado
  38. municipio
  39. codigo_postal
  40. coord_lat
  41. coord_lon
  42. version_documento
  43. hash_documento
  44. timestamp_carga
  45. usuario_captura
  46. ip_carga


In [2]:
# ── Paso 2: Diagnostico de todas las columnas ────────────────────────────────
# Cargamos todo para el diagnostico inicial
cartera = pd.read_csv('datos/cartera_polizas.csv')
mb_full = cartera.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {cartera.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in cartera.columns:
    dtype = str(cartera[col].dtype)
    nan_pct = cartera[col].isna().mean() * 100
    n_uniq  = cartera[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')

Dataset completo: (50000, 46) · 112.0 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object 

In [3]:
# ── Paso 3: Clasificar las columnas por categoria ────────────────────────────
# Esta clasificacion es una DECISION DE NEGOCIO — no solo tecnica

ANALISIS = [
    'id_poliza','num_poliza','ramo','plan','status_poliza',
    'nombre','apellido_paterno','apellido_materno',
    'rfc','edad','sexo','estado_civil','ocupacion',
    'fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia',
    'num_renovaciones','motivo_baja','fecha_nacimiento',
    'suma_asegurada','deducible','prima_neta','prima_total',
    'forma_pago','agente_id','canal_venta',
    'estado','municipio','codigo_postal',
    'marca_vehiculo','modelo_vehiculo','tipo_vehiculo',  # solo Autos
]

REDUNDANTES = [
    'nombre_completo',        # = nombre + ap_pat + ap_mat
    'id_contrato_interno',    # ≈ id_poliza
    'folio_emision',          # ≈ num_poliza
    'cuota_prima',            # = prima_total / num_cuotas
    'num_cuotas',             # derivado de forma_pago
]

ADMINISTRATIVAS = [
    'hash_documento',         # hash SHA del PDF — auditoria IT
    'timestamp_carga',        # mismo valor para todos
    'ip_carga',               # IP del servidor batch
    'usuario_captura',        # operacion interna
    'version_documento',      # version del formato del contrato
    'id_sistema_legacy',      # util SOLO para joins con sistema core
]

CONDICIONALES = [
    'nivel_educacion',        # si lo necesitas para el modelo
    'coord_lat','coord_lon',  # solo si haces analisis geoespacial
]

print(f'Analiticas:     {len(ANALISIS)}')
print(f'Redundantes:    {len(REDUNDANTES)}')
print(f'Administrativas:{len(ADMINISTRATIVAS)}')
print(f'Condicionales:  {len(CONDICIONALES)}')
print(f'Total:          {len(ANALISIS)+len(REDUNDANTES)+len(ADMINISTRATIVAS)+len(CONDICIONALES)}')

Analiticas:     32
Redundantes:    5
Administrativas:6
Condicionales:  3
Total:          46


In [4]:
# ── Paso 4: Cargar SOLO las columnas que necesitamos ─────────────────────────
t0 = time.time()
cartera_limp = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALISIS,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = cartera_limp.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALISIS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {cartera_limp.shape}')
cartera_limp.head(3)

=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (32 cols):  73.7 MB
Reduccion de memoria:      34%
Tiempo de carga:           315 ms

Dataset de trabajo: (50000, 32)


,id_poliza,num_poliza,nombre,apellido_paterno,apellido_materno,rfc,fecha_nacimiento,edad,sexo,estado_civil,...,tipo_vehiculo,suma_asegurada,deducible,prima_neta,prima_total,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,Vid-21-000001,Gabriela,Moreno,Vega,MOGV020429CG6,29/04/2002,24,F,Union libre,...,NaN,3000000,NaN,54000.0,67651.20,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,Aut-19-000002,Valeria,Torres,Castillo,TOVC020815IA8,15/08/2002,23,Femenino,Casado,...,Compacto,150000,8000.0,5250.0,6394.50,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,GMM-22-000003,Fernanda,Ramos,Silva,RAFS941018BC1,18/10/1994,31,M,Union libre,...,NaN,800000,5000.0,17600.0,22049.28,Mensual,AG051,Baja California,Tecate,45784.0


### 📝 Ejercicio 1 — Auditoria de columnas (8 min)

Usando el perfil que generaste arriba:
- **1a.** Identifica las 3 columnas con mas NaN — ¿tienen sentido esos NaN o son errores?
- **1b.** Encuentra columnas con menos de 5 valores unicos — ¿cuales deberian ser `category`?
- **1c.** Hay una columna numerica con un rango imposible (negativo o muy alto) — ¿cual es?
  *Pista: usa `df.describe()` sobre las columnas analiticas*

In [5]:
# Tu codigo aqui:
columnas_nan = cartera_limp.isna().sum().sort_values(ascending= False)
print(columnas_nan.head(3))


motivo_baja       47535
marca_vehiculo    35248
tipo_vehiculo     35248
dtype: int64


---
### Duda 2: Texto Sucio — str Operations

El campo `sexo` tiene 4 representaciones distintas del mismo valor.
Si no lo normalizas, `groupby('sexo')` produce 8 grupos en lugar de 2.

In [6]:
# ── Ver el problema ──────────────────────────────────────────────────────────
print('Valores unicos en sexo ANTES de limpiar:')
print(cartera_limp['sexo'].value_counts(dropna=False))
print(f'Total valores distintos: {cartera_limp["sexo"].nunique()}')
print()

# Si haces groupby ahora, obtienes grupos incorrectos:
grupos_incorrectos = cartera_limp.groupby('sexo')['prima_total'].mean()
print(f'Grupos sin limpiar: {len(grupos_incorrectos)} (deberian ser 2)')

Valores unicos en sexo ANTES de limpiar:
sexo
M            16697
F            16585
Femenino      8466
Masculino     8252
Name: count, dtype: int64
Total valores distintos: 4

Grupos sin limpiar: 4 (deberian ser 2)


In [7]:
# ── Solucion: normalizar con str operations ──────────────────────────────────

# Paso 1: normalizar a mayusculas sin espacios
cartera_limp['sexo'] = cartera_limp['sexo'].str.strip().str.upper()

# Paso 2: mapear todas las variantes al estandar
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',
}
cartera_limp['sexo'] = cartera_limp['sexo'].map(MAPA_SEXO)
# Valores no reconocidos quedan como NaN automaticamente

print('DESPUES de normalizar:')
print(cartera_limp['sexo'].value_counts(dropna=False))
print()
# Ahora groupby correcto
print('Prima promedio por sexo:')
print(cartera_limp.groupby('sexo')['prima_total'].mean().round(2))

DESPUES de normalizar:
sexo
F    25051
M    24949
Name: count, dtype: int64

Prima promedio por sexo:
sexo
F    29138.45
M    29428.49
Name: prima_total, dtype: float64


In [8]:
# ── Mas str operations sobre los datos reales ────────────────────────────────

# Limpiar codigo_postal: eliminar 'N/D' (ya convertido a NaN por na_values)
# Verificar:
print(f'CP con NaN: {cartera_limp["codigo_postal"].isna().sum()}')
# Rellenar CP desconocido con 'DESCONOCIDO' para no perder la fila
cartera_limp['codigo_postal'] = cartera_limp['codigo_postal'].fillna('DESCONOCIDO')

# Extraer ramo y anio desde num_poliza ('GMM-24-000123')
cartera_limp['ramo_codigo']  = cartera_limp['num_poliza'].str.extract(r'^([A-Z]+)-')
cartera_limp['anio_poliza']  = cartera_limp['num_poliza'].str.extract(r'-([0-9]{2})-').astype(float).astype('Int16')

# Verificar
print(cartera_limp[['num_poliza','ramo_codigo','anio_poliza']].head(5).to_string(index=False))

CP con NaN: 1500
   num_poliza ramo_codigo  anio_poliza
Vid-21-000001         NaN           21
Aut-19-000002         NaN           19
GMM-22-000003         GMM           22
Vid-19-000004         NaN           19
Vid-20-000005         NaN           20


---
### Duda 3: Fechas como Texto — pd.to_datetime()

In [9]:
# ── El problema: fechas llegaron como strings ────────────────────────────────
print('Tipo ANTES de convertir:', cartera_limp['fecha_nacimiento'].dtype)
print('Muestra:', cartera_limp['fecha_nacimiento'].head(3).values)
print()

# ── Convertir fecha_nacimiento (formato d/m/Y) ────────────────────────────────
cartera_limp['fecha_nacimiento'] = pd.to_datetime(
    cartera_limp['fecha_nacimiento'],
    format='%d/%m/%Y',
    errors='coerce'  # fechas invalidas → NaT (no detiene el proceso)
)

# ── Convertir fechas ISO estandar ────────────────────────────────────────────
for col_fecha in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    cartera_limp[col_fecha] = pd.to_datetime(cartera_limp[col_fecha], errors='coerce')

print('Fechas convertidas:')
print(cartera_limp[['fecha_nacimiento','fecha_emision','fecha_fin_vigencia']].dtypes)
print(f'NaT en fecha_nacimiento: {cartera_limp["fecha_nacimiento"].isna().sum()}')

Tipo ANTES de convertir: object
Muestra: ['29/04/2002' '15/08/2002' '18/10/1994']

Fechas convertidas:
fecha_nacimiento      datetime64[ns]
fecha_emision         datetime64[ns]
fecha_fin_vigencia    datetime64[ns]
dtype: object
NaT en fecha_nacimiento: 0


In [10]:
# ── Calculos actuariales con las fechas convertidas ──────────────────────────
hoy = pd.Timestamp.today()

# Edad calculada (mas precisa que la columna 'edad' del CSV)
cartera_limp['edad_calc'] = ((hoy - cartera_limp['fecha_nacimiento']).dt.days / 365.25)

# Dias de vigencia de la poliza
cartera_limp['dias_vigencia'] = (cartera_limp['fecha_fin_vigencia'] - cartera_limp['fecha_inicio_vigencia']).dt.days

# Fraccion expuesta (cuanto del periodo ya transcurrio)
dias_transcurridos = (hoy - cartera_limp['fecha_inicio_vigencia']).dt.days
cartera_limp['fraccion_expuesta'] = (dias_transcurridos / cartera_limp['dias_vigencia']).clip(0, 1).round(4)

# Componentes de fecha para groupby temporal
cartera_limp['anio_emision']     = cartera_limp['fecha_emision'].dt.year
cartera_limp['mes_emision']      = cartera_limp['fecha_emision'].dt.month
cartera_limp['trimestre_emision']= cartera_limp['fecha_emision'].dt.quarter

# Verificar
print(cartera_limp[['nombre','edad','edad_calc','dias_vigencia','fraccion_expuesta']].head(5).to_string(index=False))

  nombre  edad  edad_calc  dias_vigencia  fraccion_expuesta
Gabriela    24  24.043806            365                1.0
 Valeria    23  23.748118            366                1.0
Fernanda    31  31.572895            365                1.0
  Silvia    40  40.273785            366                1.0
 Antonio    29  29.812457            365                1.0


### 📝 Ejercicio 2 — Limpiar siniestros.csv (10 min)

El archivo `siniestros.csv` tiene fechas en 3 formatos distintos:
- `fecha_ocurrencia`: YYYY-MM-DD
- `fecha_apertura`: d/m/Y (mismo dia que fecha_reporte pero formato distinto — es REDUNDANTE)
- `fecha_ultimo_movimiento`: d/m/Y

**2a.** Carga siniestros.csv usando SOLO las columnas utiles (descarta las administrativas y redundantes).
**2b.** Convierte las 3 columnas de fecha a datetime con el formato correcto.
**2c.** Calcula `dias_reporte_real` = fecha_reporte - fecha_ocurrencia.
**2d.** Calcula `dias_resolucion_real` = fecha_cierre - fecha_reporte (NaT si no esta cerrado).
**2e.** Verifica: ¿cuantos siniestros llevan mas de 180 dias sin cerrar?

In [11]:
# Tu codigo aqui:
siniestros_cols_utiles = [
    'id_siniestro','id_poliza','ramo','tipo_siniestro',
    'fecha_ocurrencia','fecha_reporte','fecha_ultimo_movimiento','fecha_cierre',
    'dias_reporte','monto_reclamado','monto_pagado',
    'status_siniestro','motivo_rechazo','id_ajustador','fecha_apertura'
]
# Carga, convierte fechas y calcula los campos derivados:

df_sin = pd.read_csv(
    'datos/siniestros.csv',
    usecols= siniestros_cols_utiles,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
for col_fech in ['fecha_ocurrencia','fecha_apertura','fecha_ultimo_movimiento','fecha_reporte','fecha_cierre'] :
    df_sin[col_fech]=pd.to_datetime(df_sin[col_fech],
                                          format = '%d/%m/%y',
                                          errors= 'coerce')
    
df_sin['dias_reporte_real'] = (df_sin['fecha_reporte']-df_sin['fecha_ocurrencia']).dt.days
df_sin['dias_resolucion_real'] = (df_sin['fecha_cierre']-df_sin['fecha_reporte']).dt.days



---
### Duda 4: JSON — Datos de API


In [12]:
# ── Simular una respuesta JSON de una API de reaseguro ───────────────────────
import json

# Este es el tipo de JSON que recibirias de una API REST
respuesta_api = {
    'timestamp': '2026-05-02T07:00:00',
    'origen': 'Sistema_Reaseguro_v3.1',
    'polizas_reaseguradas': [
        {'id_poliza':'POL-000001','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':960.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
        {'id_poliza':'POL-000002','ramo':'Vida',
         'reasegurador':{'nombre':'Swiss Re','participacion':0.35,'prima':1820.0},
         'limites':{'maximo':5_000_000,'retencion':1_000_000}},
        {'id_poliza':'POL-000005','ramo':'GMM',
         'reasegurador':{'nombre':'Munich Re','participacion':0.40,'prima':550.0},
         'limites':{'maximo':2_000_000,'retencion':500_000}},
    ]
}

# Guardar como JSON (simula lo que llegaria de la API)
with open('datos/respuesta_reaseguro.json','w',encoding='utf-8') as f:
    json.dump(respuesta_api, f, ensure_ascii=False, indent=2)

print('JSON guardado en datos/respuesta_reaseguro.json')
print('Primeras lineas:')
print(json.dumps(respuesta_api, indent=2, ensure_ascii=False)[:300])

JSON guardado en datos/respuesta_reaseguro.json
Primeras lineas:
{
  "timestamp": "2026-05-02T07:00:00",
  "origen": "Sistema_Reaseguro_v3.1",
  "polizas_reaseguradas": [
    {
      "id_poliza": "POL-000001",
      "ramo": "GMM",
      "reasegurador": {
        "nombre": "Munich Re",
        "participacion": 0.4,
        "prima": 960.0
      },
      "limites": 


In [13]:
# ── Problema: JSON anidado no se carga directo en DataFrame ─────────────────
# pd.read_json funciona para JSON simple pero no para estructuras anidadas

# Cargar el JSON
with open('datos/respuesta_reaseguro.json') as f:
    data = json.load(f)

# Intentar con pd.read_json — no da el resultado esperado con anidados
# pd.read_json('datos/respuesta_reaseguro.json')  # solo lee nivel 1

# ── Solucion: json_normalize aplana el JSON anidado ──────────────────────────
from pandas import json_normalize

df_reas = json_normalize(
    data['polizas_reaseguradas'],
    sep='_'    # separador para campos anidados
)

print('DataFrame aplanado:')
print(df_reas.to_string(index=False))
print()
print('Columnas generadas:', list(df_reas.columns))

DataFrame aplanado:
 id_poliza ramo reasegurador_nombre  reasegurador_participacion  reasegurador_prima  limites_maximo  limites_retencion
POL-000001  GMM           Munich Re                        0.40               960.0         2000000             500000
POL-000002 Vida            Swiss Re                        0.35              1820.0         5000000            1000000
POL-000005  GMM           Munich Re                        0.40               550.0         2000000             500000

Columnas generadas: ['id_poliza', 'ramo', 'reasegurador_nombre', 'reasegurador_participacion', 'reasegurador_prima', 'limites_maximo', 'limites_retencion']


In [14]:
# ── json_normalize con listas anidadas ───────────────────────────────────────
# Caso mas complejo: cuando hay listas dentro del JSON

respuesta_multi = {
    'polizas': [
        {'id':'P01','coberturas':[{'tipo':'Hospitalizacion','suma':500_000},{'tipo':'Cirugia','suma':300_000}]},
        {'id':'P02','coberturas':[{'tipo':'Hospitalizacion','suma':800_000}]},
    ]
}

# record_path: donde esta la lista a 'explotar'
# meta: campos del nivel padre que quieres conservar
df_cob = json_normalize(
    respuesta_multi['polizas'],
    record_path='coberturas',
    meta=['id'],
    sep='_'
)
print(df_cob)

              tipo    suma   id
0  Hospitalizacion  500000  P01
1          Cirugia  300000  P01
2  Hospitalizacion  800000  P02


In [15]:
# ── Guardar DataFrame como JSON ──────────────────────────────────────────────

# orient='records' — lista de objetos (lo mas comun para APIs)
cartera_limp.head(10).to_json('datos/muestra.json',
    orient='records',
    force_ascii=False,  # preserva caracteres especiales (acentos)
    indent=2,
    date_format='iso'   # fechas en formato ISO
)

# Verificar
with open('datos/muestra.json') as f:
    preview = f.read(300)
print(preview)

[
  {
    "id_poliza":"POL-000001",
    "num_poliza":"Vid-21-000001",
    "nombre":"Gabriela",
    "apellido_paterno":"Moreno",
    "apellido_materno":"Vega",
    "rfc":"MOGV020429CG6",
    "fecha_nacimiento":"2002-04-29T00:00:00.000",
    "edad":24,
    "sexo":"F",
    "estado_civil":"Union libre",


---
### Duda 5: 90k Filas — Procesar sin Esperar (chunks)

In [16]:
# ── Ver el tamano del archivo de beneficiarios ───────────────────────────────
mb_ben = os.path.getsize('datos/beneficiarios.csv') / 1024**2
print(f'beneficiarios.csv: {mb_ben:.1f} MB')

# Contar filas sin cargar todo
total_ben = sum(len(chunk) for chunk in
    pd.read_csv('datos/beneficiarios.csv', chunksize=10_000))
print(f'Total beneficiarios: {total_ben:,}')

# ── Patron real: calcular estadisticas por parentesco ─────────────────────────
# Sin cargar los 90k en memoria a la vez
conteo_parentesco = {}

for chunk in pd.read_csv('datos/beneficiarios.csv', chunksize=10_000):
    counts = chunk['parentesco'].value_counts().to_dict()
    for k, v in counts.items():
        conteo_parentesco[k] = conteo_parentesco.get(k, 0) + v

resultado = pd.Series(conteo_parentesco).sort_values(ascending=False)
print('Beneficiarios por parentesco (procesado por chunks):')
print(resultado)

beneficiarios.csv: 11.5 MB
Total beneficiarios: 90,000
Beneficiarios por parentesco (procesado por chunks):
Conyuge    26718
Hijo       22785
Hija       17871
Madre       7298
Padre       7173
Hermano     3596
Hermana     2765
Otro        1794
dtype: int64


In [17]:
# ── Filtrar y concatenar solo lo que necesitas ───────────────────────────────
# Obtener solo beneficiarios activos de polizas de Vida

partes = []
for chunk in pd.read_csv('datos/beneficiarios.csv',
                         chunksize=10_000,
                         usecols=['id_beneficiario','id_poliza','nombre',
                                  'apellido_paterno','parentesco','porcentaje','activo']):
    filtrado = chunk[chunk['activo'] == True]
    if len(filtrado) > 0:
        partes.append(filtrado)

ben_activos = pd.concat(partes, ignore_index=True)
print(f'Beneficiarios activos: {len(ben_activos):,}')
print(f'Memoria: {ben_activos.memory_usage(deep=True).sum()/1024:.0f} KB')

Beneficiarios activos: 67,656
Memoria: 21956 KB


---
### Duda 6: Downcast — Cuándo Es Seguro

In [18]:
# ── Demostrar el problema de precision ───────────────────────────────────────
import numpy as np

# float64 vs float32 con valores grandes
prima_grande = 15_432_756.80
print(f'Original (float64): {prima_grande}')
print(f'Como float32:       {np.float32(prima_grande)}')
print(f'Diferencia:         {abs(prima_grande - float(np.float32(prima_grande))):.2f}')
print()

# Con valores tipicos de primas individuales
prima_gmm = 3_450.75
print(f'Prima GMM (float64): {prima_gmm}')
print(f'Prima GMM (float32): {np.float32(prima_gmm)}')
print(f'Diferencia:          {abs(prima_gmm - float(np.float32(prima_gmm))):.6f}')

Original (float64): 15432756.8
Como float32:       15432757.0
Diferencia:         0.20

Prima GMM (float64): 3450.75
Prima GMM (float32): 3450.75
Diferencia:          0.000000


In [19]:
# ── Estrategia segura de optimizacion ────────────────────────────────────────
print('Memoria ANTES de optimizar:')
print(f'{cartera_limp.memory_usage(deep=True).sum()/1024**2:.2f} MB')
print()

df_opt = cartera_limp.copy()

# SEGURO: categoricas para columnas con pocos valores unicos
cols_category = ['ramo','plan','status_poliza','sexo','canal_venta',
                 'forma_pago','estado','estado_civil','tipo_vehiculo']
for col in cols_category:
    if col in df_opt.columns:
        n_uniq = df_opt[col].nunique()
        n_tot  = len(df_opt)
        pct = n_uniq/n_tot
        print(f'  {col:<25}: {n_uniq:>5} unicos ({pct:.1%}) → category')
        df_opt[col] = df_opt[col].astype('category')

# SEGURO: enteros pequenos
df_opt['num_renovaciones'] = df_opt['num_renovaciones'].fillna(0).astype('int8')

# SEGURO: booleano
# (activa ya no esta porque la descartamos, pero aplica el principio)

# CONSERVAR en float64: primas y sumas (montos grandes o con centavos importantes)
# NO hacer: df_opt['prima_total'] = df_opt['prima_total'].astype('float32')

print()
print('Memoria DESPUES de optimizar:')
mb_antes = cartera_limp.memory_usage(deep=True).sum()/1024**2
mb_desp  = df_opt.memory_usage(deep=True).sum()/1024**2
print(f'{mb_antes:.2f} MB → {mb_desp:.2f} MB ({(1-mb_desp/mb_antes)*100:.0f}% reduccion)')

Memoria ANTES de optimizar:
67.51 MB

  ramo                     :     4 unicos (0.0%) → category
  plan                     :    12 unicos (0.0%) → category
  status_poliza            :     4 unicos (0.0%) → category
  sexo                     :     2 unicos (0.0%) → category
  canal_venta              :     6 unicos (0.0%) → category
  forma_pago               :     4 unicos (0.0%) → category
  estado                   :    15 unicos (0.0%) → category
  estado_civil             :     5 unicos (0.0%) → category
  tipo_vehiculo            :     8 unicos (0.0%) → category

Memoria DESPUES de optimizar:
67.51 MB → 41.45 MB (39% reduccion)


### 📝 Ejercicio 3 — Pipeline de limpieza completo (12 min)

Encapsula todo lo que hicimos en una funcion `limpiar_cartera(ruta_csv)` que:
- Carga con `usecols=ANALITICAS`
- Normaliza `sexo` con str + map
- Convierte fechas con `to_datetime` + `errors='coerce'`
- Calcula `edad_calc`, `dias_vigencia`, `fraccion_expuesta`
- Aplica optimizacion de memoria (categoricas)
- Retorna el DataFrame limpio

Al final llama: `df_limpio = limpiar_cartera('datos/cartera_polizas.csv')`
y verifica que no tenga texto sucio en sexo.

In [20]:
# Tu codigo aqui:
def limpiar_cartera(ruta_csv):
    # Implementa la funcion completa aqui
    data = pd.read_csv(ruta_csv, usecols= ANALISIS,
                       na_values= ['N/D','N/A','ND','--','Sin dato',''],)
    data['sexo'] = data['sexo'].str.strip().str.upper()

    mapeo_sexo = {'M': 'M', 'MASCULINO': 'M', 'HOMBRE': 'M', 'MASC': 'M',
    'F': 'F', 'FEMENINO': 'F', 'MUJER': 'F', 'FEM': 'F',}

    data['sexo'] = data['sexo'].map(mapeo_sexo)
    pass


---
### Duda 7: ¿Cuándo Cambiar a Polars?

In [21]:
# ── Instalar polars si no esta disponible ────────────────────────────────────
# En tu terminal: pip install polars
# Verificar:
try:
    import polars as pl
    print(f'Polars {pl.__version__} disponible')
    POLARS_OK = True
except ImportError:
    print('Polars no instalado. Ejecuta: pip install polars')
    POLARS_OK = False

Polars 1.40.1 disponible


In [22]:
!pip install polars

In [23]:
# ── Comparativa de velocidad pandas vs polars ────────────────────────────────
if POLARS_OK:
    import polars as pl
    import time

    # ── Pandas ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALISIS)
    res_pd = (df_pd.groupby('ramo')
                   .agg(polizas=('id_poliza','count'),
                        prima_total=('prima_total','sum'),
                        prima_prom=('prima_neta','mean'))
                   .round(2).reset_index())
    t_pd = time.time()-t0

    # ── Polars ──────────────────────────────────────────────────────────────
    t0 = time.time()
    df_pl = pl.read_csv('datos/cartera_polizas.csv', columns=ANALISIS)
    res_pl = (df_pl
        .group_by('ramo')
        .agg([
            pl.col('id_poliza').count().alias('polizas'),
            pl.col('prima_total').sum().alias('prima_total'),
            pl.col('prima_neta').mean().alias('prima_prom'),
        ])
    )
    t_pl = time.time()-t0

    print(f'Pandas:  {t_pd*1000:.0f} ms')
    print(f'Polars:  {t_pl*1000:.0f} ms')
    print(f'Polars es {t_pd/t_pl:.1f}x mas rapido en esta operacion')
    print()
    print('Resultado Polars:')
    print(res_pl)
else:
    print('Instala polars para ver la comparativa: pip install polars')

Pandas:  355 ms
Polars:  97 ms
Polars es 3.6x mas rapido en esta operacion

Resultado Polars:
shape: (4, 4)
┌───────────────────────┬─────────┬─────────────┬──────────────┐
│ ramo                  ┆ polizas ┆ prima_total ┆ prima_prom   │
│ ---                   ┆ ---     ┆ ---         ┆ ---          │
│ str                   ┆ u32     ┆ f64         ┆ f64          │
╞═══════════════════════╪═════════╪═════════════╪══════════════╡
│ Autos                 ┆ 14752   ┆ 2.5382e8    ┆ 14349.345658 │
│ GMM                   ┆ 22531   ┆ 7.3102e8    ┆ 27065.47908  │
│ Accidentes Personales ┆ 5207    ┆ 2.4981e7    ┆ 4002.032838  │
│ Vida                  ┆ 7510    ┆ 4.5433e8    ┆ 50439.777113 │
└───────────────────────┴─────────┴─────────────┴──────────────┘


In [24]:
# ── Sintaxis Polars: las operaciones mas comunes ──────────────────────────────
if POLARS_OK:
    df_pl = pl.read_csv('datos/cartera_polizas.csv',
                        columns=['id_poliza','ramo','prima_total','edad'])

    # Filtrar
    print('Polizas con prima > 10,000:')
    print(df_pl.filter(pl.col('prima_total') > 10_000).shape)

    # Agregar columna
    df_pl2 = df_pl.with_columns(
        (pl.col('prima_total')/12).alias('prima_mensual')
    )

    # Sort
    print('Top 5 primas:')
    print(df_pl2.sort('prima_total', descending=True).head(5)[['id_poliza','ramo','prima_total']])

    # Lazy evaluation — ejecuta todo optimizado al final
    resultado = (
        pl.scan_csv('datos/cartera_polizas.csv')  # NO carga en memoria aun
        .filter(pl.col('prima_total') > 5000)
        .group_by('ramo')
        .agg(pl.col('prima_total').sum())
        .collect()  # AHORA ejecuta con el plan optimizado
    )
    print('Lazy evaluation result:')
    print(resultado)

Polizas con prima > 10,000:
(33919, 4)
Top 5 primas:
shape: (5, 3)
┌────────────┬──────┬─────────────┐
│ id_poliza  ┆ ramo ┆ prima_total │
│ ---        ┆ ---  ┆ ---         │
│ str        ┆ str  ┆ f64         │
╞════════════╪══════╪═════════════╡
│ POL-007740 ┆ Vida ┆ 182658.24   │
│ POL-007890 ┆ Vida ┆ 182658.24   │
│ POL-024748 ┆ Vida ┆ 182658.24   │
│ POL-003193 ┆ Vida ┆ 180403.2    │
│ POL-007981 ┆ Vida ┆ 180403.2    │
└────────────┴──────┴─────────────┘
Lazy evaluation result:
shape: (4, 2)
┌───────────────────────┬─────────────┐
│ ramo                  ┆ prima_total │
│ ---                   ┆ ---         │
│ str                   ┆ f64         │
╞═══════════════════════╪═════════════╡
│ GMM                   ┆ 7.3102e8    │
│ Accidentes Personales ┆ 1.2546e7    │
│ Vida                  ┆ 4.5433e8    │
│ Autos                 ┆ 2.5382e8    │
└───────────────────────┴─────────────┘


### Regla practica: ¿Pandas o Polars?

| Situacion | Usa |
|-----------|-----|
| Aprendizaje, primeros modelos | **pandas** — el ecosistema es enorme |
| < 1 millon de filas | **pandas** — mas que suficiente |
| Analisis interactivo en notebook | **pandas** — syntax mas conocida |
| Pipeline de produccion > 5M filas | **polars** — 5-20x mas rapido |
| Ingesta diaria de datos grandes | **polars** con lazy evaluation |
| ETL de empresa | **polars** o **spark** segun el tamano |

---
## pivot_table con Datos Reales


In [25]:
# ── Agregar columnas necesarias para el pivot ────────────────────────────────
df_work = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALISIS,
                      na_values=['N/D','N/A',''])
df_work['prima_neta'] = df_work['prima_neta'].fillna(df_work.groupby('ramo')['prima_neta'].transform('median'))
df_work['g_edad'] = pd.cut(df_work['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
df_work['siniest_flag'] = (df_work['prima_neta'] > df_work['prima_neta'].quantile(0.75)).astype(int)

print(f'Dataset para pivot: {df_work.shape}')

Dataset para pivot: (50000, 34)


In [26]:
# ── pivot_table: prima por ramo x grupo de edad ──────────────────────────────
tabla_prima = pd.pivot_table(
    df_work,
    values   = 'prima_total',
    index    = 'ramo',
    columns  = 'g_edad',
    aggfunc  = 'sum',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
).round(0) / 1_000  # en miles de pesos

print('Prima total por ramo y grupo de edad (miles MXN):')
print(tabla_prima.to_string())

Prima total por ramo y grupo de edad (miles MXN):
g_edad                      18-30       31-45       46-60         61+        TOTAL
ramo                                                                              
Accidentes Personales    5248.369    8348.395    8731.668    2652.247    24980.679
Autos                   55457.022   84225.512   84571.404   29569.569   253823.506
GMM                    139278.258  223908.688  264218.298  103617.078   731022.322
Vida                    79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                  279651.498  451257.236  523823.143  209426.754  1464158.631


C:\Users\julia\AppData\Local\Temp\ipykernel_31000\3938379720.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla_prima = pd.pivot_table(


In [27]:
# ── pivot_table: polizas por zona x canal ────────────────────────────────────
tabla_canal = pd.pivot_table(
    df_work,
    values   = 'id_poliza',
    index    = 'estado',
    columns  = 'canal_venta',
    aggfunc  = 'count',
    fill_value = 0,
    margins    = True,
    margins_name = 'TOTAL'
)

print('Polizas por estado y canal de venta:')
print(tabla_canal.to_string())

Polizas por estado y canal de venta:
canal_venta       Agente  Banca Seguros  Broker  Digital  Directo  Promotor  TOTAL
estado                                                                            
Baja California     1646            329     678      241      356        98   3348
CDMX                1719            311     684      217      318       108   3357
Chihuahua           1656            344     642      250      357       101   3350
Coahuila            1721            349     672      199      329        96   3366
Estado de Mexico    1703            336     652      253      287        93   3324
Guanajuato          1745            339     669      237      359        96   3445
Jalisco             1664            322     670      258      332       106   3352
Michoacan           1621            327     711      217      319        99   3294
Nuevo Leon          1681            335     615      225      326        96   3278
Puebla              1586            330     672   

---
## Exportar — CSV, Excel, Parquet y JSON


In [28]:
# ── Guardar en todos los formatos y comparar ──────────────────────────────────
df_export = df_work.head(10_000)  # subconjunto para demo

formatos = {
    'CSV':     ('datos/export_demo.csv',
                lambda: df_export.to_csv('datos/export_demo.csv', index=False)),
    'Parquet': ('datos/export_demo.parquet',
                lambda: df_export.to_parquet('datos/export_demo.parquet', index=False)),
    'JSON':    ('datos/export_demo.json',
                lambda: df_export.to_json('datos/export_demo.json',
                                          orient='records', force_ascii=False)),
}

print(f'{"Formato":<10} {"Tamano":>10} {"Tiempo":>10}')
print('-' * 35)
for nombre, (ruta, guardar) in formatos.items():
    t0 = time.time()
    guardar()
    t = (time.time()-t0)*1000
    kb = os.path.getsize(ruta)/1024
    print(f'{nombre:<10} {kb:>8.0f} KB {t:>8.0f} ms')

Formato        Tamano     Tiempo
-----------------------------------
CSV            2465 KB       95 ms
Parquet         633 KB      163 ms
JSON           7645 KB       56 ms


In [29]:
# ── Excel multihoja — el entregable mas solicitado en empresas ───────────────
resumen_ramo = df_work.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
).round(2).reset_index()
resumen_ramo['pct_cartera'] = (resumen_ramo['prima_total']/resumen_ramo['prima_total'].sum()*100).round(1)

with pd.ExcelWriter('datos/reporte_demo.xlsx', engine='openpyxl') as writer:
    df_work.head(5_000).to_excel(writer, sheet_name='Cartera', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    tabla_prima.to_excel(writer, sheet_name='Pivot_Prima')
    tabla_canal.to_excel(writer, sheet_name='Pivot_Canal')

kb_xl = os.path.getsize('datos/reporte_demo.xlsx')/1024
print(f'Excel multihoja generado: {kb_xl:.0f} KB con 4 hojas')

Excel multihoja generado: 972 KB con 4 hojas


---
##  Ejercicio Integrador Final — Pipeline Completo

**Contexto:** Tu jefa de estadistica te pide el reporte ejecutivo del Q1 2026.
Tienes los 4 archivos crudos. Debes construir un pipeline completo,
documentando cada decision de limpieza.

**Tiempo:** 40 minutos  |  **Entregable:** Excel con 5 hojas + Parquet

---

### Criterios de evaluacion:
- ✅ Cada decision de descarte de columnas esta justificada con comentario
- ✅ No hay texto sucio en columnas categoricas clave
- ✅ Todas las fechas son datetime, no object
- ✅ dtypes optimizados sin perder precision en montos
- ✅ El Excel tiene las 5 hojas con los datos correctos
- ✅ El Parquet es mas pequeno que el CSV equivalente

In [142]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 1: INGESTA INTELIGENTE
# ══════════════════════════════════════════════════════════════════════════════
# Carga cartera, siniestros y catalogo de ramos/agentes.
# Usa SOLO las columnas que necesitas — justifica con comentario.
# Mide la memoria ahorrada vs cargar todo.

# Tu codigo aqui:
cartera = pd.read_csv('datos/cartera_polizas.csv')

mb_full = cartera.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset completo: {cartera.shape} · {mb_full:.1f} MB')
print()

# Perfil de cada columna: tipo, NaN%, valores unicos
print(f'{"Columna":<30} {"Dtype":<12} {"NaN%":>7} {"Unicos":>8}')
print('-' * 65)
for col in cartera.columns:
    dtype = str(cartera[col].dtype)
    nan_pct = cartera[col].isna().mean() * 100
    n_uniq  = cartera[col].nunique()
    print(f'{col:<30} {dtype:<12} {nan_pct:>6.1f}% {n_uniq:>8,}')



Dataset completo: (50000, 46) · 112.0 MB

Columna                        Dtype           NaN%   Unicos
-----------------------------------------------------------------
id_poliza                      object          0.0%   50,000
num_poliza                     object          0.0%   50,000
id_contrato_interno            object          0.0%   48,572
folio_emision                  object          0.0%   49,870
id_sistema_legacy              object          0.0%   49,988
nombre                         object          0.0%       55
apellido_paterno               object          0.0%       38
apellido_materno               object          0.0%       23
nombre_completo                object          0.0%   31,140
rfc                            object          0.0%   50,000
fecha_nacimiento               object          0.0%   15,676
edad                           int64           0.0%       46
sexo                           object          0.0%        4
estado_civil                   object 

## Columnas a utilizar

Las columnas a utilizar seran: 
- ✅id_poliza : Para tener un identificador de la Póliza para obtener información derivada, como siniestros y vigencia.
- ✅fecha_nacimiento : Es importante para temas legales y el cálculo de columnas dérivadas y para tarificación.
- ✅sexo : Segregación de riesgos, así como tarificación de GMM y Vida.
- ✅Ocupación: Extraprimar o medir distintos tipos de Riesgos.
- ✅nivel_educacion: Para futuros análisis.
- ✅ramo: segregar pólizas.
- ✅plan: Análisis de costos, siniestralidad, cambio de condiciones.
- ✅fecha_emision, fecha_inicio_vigencia, fecha_fin_vigencia : Calcular devengamientos, vigencias, renovaciones, etc.
- ✅num_renovaciones: Ver cuanto tiempo lleva la póliza con un grupo para así ofertar descuentos o tener un mejor análsis de la siniestralidad.
- ✅status_poliza: Ver si está activa la póliza para ver si vale la pena ser analizada.
- ✅motivo_baja: Analisis para retención de negocio.
- ✅ marca_vehiculo,modelo_vehiculo,tipo_vehiculo : Análisis del ramo de auto.
- ✅suma_asegurada,deducible,prima_neta,prima_total,cuota_prima: Análisis de siniestralidad.
- ✅forma_pago : Cálculo de ingreso de prima.
- ✅estado, municipio : Análisis de Riesgos.



In [143]:
# Hacemos nuestro data frame con las columnas que utilizarermos

ANALISIS = ['id_poliza', 'fecha_nacimiento', 'sexo', 'ocupacion', 'nivel_educacion', 'ramo', 'plan',
'fecha_emision', 'fecha_inicio_vigencia', 'fecha_fin_vigencia', 'num_renovaciones', 'status_poliza', 
'motivo_baja', 'marca_vehiculo', 'modelo_vehiculo', 'tipo_vehiculo', 'suma_asegurada', 'deducible', 
'prima_neta', 'prima_total', 'cuota_prima', 'forma_pago', 'estado', 'municipio', 'codigo_postal','agente_id','edad']

t0 = time.time()
cartera_limpia = pd.read_csv(
    'datos/cartera_polizas.csv',
    usecols=ANALISIS,
    na_values=['N/D','N/A','ND','--','Sin dato',''],
)
t1 = time.time()
mb_opt = cartera_limpia.memory_usage(deep=True).sum() / 1024**2

print('=== COMPARATIVA DE CARGA ===')
print(f'Carga completa (46 cols):  {mb_full:.1f} MB')
print(f'Carga selectiva ({len(ANALISIS)} cols):  {mb_opt:.1f} MB')
print(f'Reduccion de memoria:      {(1-mb_opt/mb_full)*100:.0f}%')
print(f'Tiempo de carga:           {(t1-t0)*1000:.0f} ms')
print()
print(f'Dataset de trabajo: {cartera_limpia.shape}')
cartera_limpia.head(3)

=== COMPARATIVA DE CARGA ===
Carga completa (46 cols):  112.0 MB
Carga selectiva (27 cols):  55.4 MB
Reduccion de memoria:      51%
Tiempo de carga:           236 ms

Dataset de trabajo: (50000, 27)


,id_poliza,fecha_nacimiento,edad,sexo,ocupacion,nivel_educacion,ramo,plan,fecha_emision,fecha_inicio_vigencia,...,suma_asegurada,deducible,prima_neta,prima_total,cuota_prima,forma_pago,agente_id,estado,municipio,codigo_postal
0,POL-000001,29/04/2002,24,F,Independiente,Licenciatura,Vida,10 anios,2021-11-23,2021-11-23,...,3000000,NaN,54000.0,67651.20,5637.60,Mensual,AG054,Veracruz,Poza Rica,36619.0
1,POL-000002,15/08/2002,23,Femenino,Contador,Licenciatura,Autos,Amplia Plus,2019-08-19,2019-08-19,...,150000,8000.0,5250.0,6394.50,1598.62,Trimestral,AG004,Michoacan,Morelia,58889.0
2,POL-000003,18/10/1994,31,M,Ingeniero,Licenciatura,GMM,Basico,2022-07-11,2022-07-11,...,800000,5000.0,17600.0,22049.28,1837.44,Mensual,AG051,Baja California,Tecate,45784.0


In [144]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 2: LIMPIEZA COMPLETA
# ══════════════════════════════════════════════════════════════════════════════
# 2a. Elimina duplicados de cartera
# 2b. Normaliza sexo con str.strip().str.upper() + .map(MAPA_SEXO)
# 2c. Convierte TODAS las fechas a datetime con errors='coerce'
# 2d. Rellena NaN de prima_neta con la MEDIANA POR RAMO (no global)
#     df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# 2e. Limpia codigo_postal (reemplaza 'N/D' con NaN)
# 2f. Aplica optimizacion de categoricas (sin tocar float64 de primas)

# Tu codigo aqui:
print("="*50)
print('ANTES DEL MAPEO')
cartera_limpia = cartera_limpia.drop_duplicates(subset='id_poliza') # quitamos duplicados basandonos en el id de poliza, que es unico por contrato.

dif_sexo = cartera_limpia.groupby('sexo')['sexo'].count() # vemos cuantas categorias distintas de sexo hay

print(dif_sexo)

# Hacemos el mapeo de sexo para normalizarlo
MAPA_SEXO = {
    'M': 'M', 'MASCULINO': 'M', 'F':'F', 'FEMENINO': 'F'}

cartera_limpia['sexo'] = cartera_limpia['sexo'].str.strip().str.upper().map(MAPA_SEXO)
print("="*50)
print('DESPUÉS DEL MAPEO')
print(cartera_limpia['sexo'].value_counts(dropna=False))

ANTES DEL MAPEO
sexo
F            16585
Femenino      8466
M            16697
Masculino     8252
Name: sexo, dtype: int64
DESPUÉS DEL MAPEO
sexo
F    25051
M    24949
Name: count, dtype: int64


In [145]:
fechas=cartera_limpia[['fecha_nacimiento','fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']]
fechas.head(3)

,fecha_nacimiento,fecha_emision,fecha_inicio_vigencia,fecha_fin_vigencia
0,29/04/2002,2021-11-23,2021-11-23,2022-11-23
1,15/08/2002,2019-08-19,2019-08-19,2020-08-19
2,18/10/1994,2022-07-11,2022-07-11,2023-07-11


In [146]:
# ── vemos el formato de las fechas ────────────────────────────────
for col in ['fecha_nacimiento','fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    print(f'Tipo ANTES de convertir {col}:', cartera_limpia[col].dtype)
 
print("="*50)
cartera_limpia['fecha_nacimiento'] = pd.to_datetime(
    cartera_limpia['fecha_nacimiento'], format='%d/%m/%Y', errors='coerce')

for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
        cartera_limpia[col] = pd.to_datetime(cartera_limpia[col], errors='coerce')

for col in ['fecha_nacimiento','fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
    print(f'Tipo DESPUES de convertir {col}:', cartera_limpia[col].dtype)

Tipo ANTES de convertir fecha_nacimiento: object
Tipo ANTES de convertir fecha_emision: object
Tipo ANTES de convertir fecha_inicio_vigencia: object
Tipo ANTES de convertir fecha_fin_vigencia: object
Tipo DESPUES de convertir fecha_nacimiento: datetime64[ns]
Tipo DESPUES de convertir fecha_emision: datetime64[ns]
Tipo DESPUES de convertir fecha_inicio_vigencia: datetime64[ns]
Tipo DESPUES de convertir fecha_fin_vigencia: datetime64[ns]


In [147]:
fechas=cartera_limpia[['fecha_nacimiento','fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']]
fechas.head(3)

,fecha_nacimiento,fecha_emision,fecha_inicio_vigencia,fecha_fin_vigencia
0,2002-04-29,2021-11-23,2021-11-23,2022-11-23
1,2002-08-15,2019-08-19,2019-08-19,2020-08-19
2,1994-10-18,2022-07-11,2022-07-11,2023-07-11


In [148]:
# rellenamos los valores nulos de prima_neta con la mediana por ramo
cartera_limpia['prima_neta'] = cartera_limpia.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))   
nas_prima_neta = cartera_limpia['prima_neta'].isna().sum()
print(f'NaN en prima_neta DESPUES de rellenar con mediana por ramo: {nas_prima_neta}')

## REEMPLAZAMOS 'N/D' EN CODIGO POSTAL CON NaN
cartera_limpia['codigo_postal'] = cartera_limpia['codigo_postal'].replace('N/D', np.nan)
## LIMPIAMOS LAS CATEGORICAS

 # Categoricas
for col in  cartera_limpia.select_dtypes('object').columns:
    pct = cartera_limpia[col].nunique() / len(cartera_limpia)
    if pct < 0.05:  # si menos del 5% de valores son unicos, es candidata a category
        cartera_limpia[col] = cartera_limpia[col].astype('category')

    # Enteros: downcast seguro
for col in cartera_limpia.select_dtypes('int64').columns:
    mx = cartera_limpia[col].abs().max()
    if mx <= 127:    cartera_limpia[col] = cartera_limpia[col].astype('int8')
    elif mx <= 32767:cartera_limpia[col] = cartera_limpia[col].astype('int16')
    elif mx <= 2_147_483_647: cartera_limpia[col] = cartera_limpia[col].astype('int32')

NaN en prima_neta DESPUES de rellenar con mediana por ramo: 0


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 3: ENRIQUECIMIENTO
# ══════════════════════════════════════════════════════════════════════════════
# 3a. Merge con catalogo_ramos: agregar nombre_largo, tasa_base
# 3b. Merge con catalogo_agentes: agregar nombre del agente, region
# 3c. Crear: g_edad con pd.cut
# 3d. Crear: prima_calc = suma_asegurada * tasa_base * 1.16
# 3e. Crear: nivel_riesgo con .apply(clasificar_riesgo) — de mi_modulo
# 3f. Crear: edad_calc desde fecha_nacimiento
# 3g. Crear: dias_vigencia, fraccion_expuesta

# Tu codigo aqui:
#Cargamos lAS TABLAS QUE VAMOS A USAR PARA HACER LOS CRUCES
catalogo_ramos = pd.read_csv('datos/catalogo_ramos.csv')
catalogo_agentes = pd.read_csv('datos/catalogo_agentes.csv')
#JUNTAMOS CON EL CATALOGO DE RAMOS PARA OBTENER EL NOMBRE LARGO Y LA TASA BASE
cartera_limpia = pd.merge(cartera_limpia, catalogo_ramos[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
#JUNTAMOS CON EL CATALOGO DE AGENTES PARA OBTENER EL NOMBRE DEL AGENTE, LA REGION Y EL TIPO DE AGENTE
cartera_limpia = pd.merge(cartera_limpia, catalogo_agentes[['agente_id','nombre','region','tipo']].rename(
                  columns={'nombre':'nombre_agente','tipo':'tipo_agente'}),
                  on='agente_id', how='left')
#CREAMOS LA COLUMNA DE GRUPOS DE EDAD
hoy = pd.Timestamp.today()
cartera_limpia['edad_calc'] = (hoy - cartera_limpia['fecha_nacimiento']).dt.days / 365.25
cartera_limpia['g_edad'] = pd.cut(cartera_limpia['edad'], bins=[0,30,45,60,100],
                           labels=['18-30','31-45','46-60','61+'])
#creamoas la columna de prima calculada
cartera_limpia['prima_calc'] = cartera_limpia['suma_asegurada'] * cartera_limpia['tasa_base'] * 1.16
#importamos la funcion de clasificacion de riesgo de mi_modulo
from mi_modulo import clasificar_riesgo
#Creamos aleatoriamente el numero de siniestros para cada poliza para poder clasificar el nivel de riesgo
np.random.seed(42)
cartera_limpia['num_siniestros'] = np.random.randint(0, 5, size=len(cartera_limpia))
cartera_limpia['nivel_riesgo'] = cartera_limpia['num_siniestros'].apply(clasificar_riesgo)
#calculamos los dias de vigencia y la fraccion expuesta
cartera_limpia['dias_vigencia'] = (cartera_limpia['fecha_fin_vigencia'] - cartera_limpia['fecha_inicio_vigencia']).dt.days
dias_transcurridos = (hoy - cartera_limpia['fecha_inicio_vigencia']).dt.days
cartera_limpia['fraccion_expuesta'] = (dias_transcurridos / cartera_limpia['dias_vigencia']).clip(0, 1).round(4)



,id_poliza,fecha_nacimiento,edad,sexo,ocupacion,nivel_educacion,ramo,plan,fecha_emision,fecha_inicio_vigencia,...,nombre_agente,region,tipo_agente,edad_calc,g_edad,prima_calc,num_siniestros,nivel_riesgo,dias_vigencia,fraccion_expuesta
0,POL-000001,2002-04-29,24,F,Independiente,Licenciatura,Vida,10 anios,2021-11-23,2021-11-23,...,Enrique Castillo,Chihuahua,Independiente,24.043806,18-30,62640.0,3,ALTO,365,1.0
1,POL-000002,2002-08-15,23,F,Contador,Licenciatura,Autos,Amplia Plus,2019-08-19,2019-08-19,...,Patricia Rios,Veracruz,Independiente,23.748118,18-30,6090.0,4,ALTO,366,1.0
2,POL-000003,1994-10-18,31,M,Ingeniero,Licenciatura,GMM,Basico,2022-07-11,2022-07-11,...,Manuel Mendoza,Chihuahua,Independiente,31.572895,31-45,20416.0,2,MEDIO,365,1.0


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 4: ANALISIS Y REPORTES
# ══════════════════════════════════════════════════════════════════════════════
# 4a. groupby+agg por ramo: polizas, prima_total, prima_prom, pct_cartera
# 4b. groupby+agg por agente: polizas, prima_total, comision (10%)
# 4c. pivot_table prima por ramo x g_edad con margins=True
# 4d. pivot_table polizas por estado x ramo
# 4e. Identifica: ramo con mayor prima total y zona con mayor frecuencia

# Tu codigo aqui:
#agrupamos por ramo para obtener el numero de polizas, la prima total, la prima promedio y el porcentaje 
# que representa cada ramo del total de la cartera
resumen_ramo = cartera_limpia.groupby('ramo').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    prima_prom =('prima_total','mean'),
    pct_cartera =('prima_total', lambda x: x.sum()/cartera_limpia['prima_total'].sum()*100)
).round(2).reset_index().sort_values('prima_total', ascending=False)
# Mostramos nuestro resumen por ramo
print('Resumen por ramo:')
print(resumen_ramo[['ramo','polizas','prima_total','pct_cartera']].to_string(index=False))

# Hacemos resumen por agente, con el numero de polizas, la prima total y la comision (10% de la prima total)
resumen_agente = cartera_limpia.groupby('nombre_agente').agg(
    polizas    =('id_poliza','count'),
    prima_total=('prima_total','sum'),
    comision    =('prima_total', lambda x: x.sum()*0.10)
).round(2).reset_index().sort_values('prima_total', ascending=False)
# Mostramos nuestro resumen por agente
print("="*65)
print('Resumen por agente:')
print(resumen_agente[['nombre_agente','polizas','prima_total','comision']].to_string(index=False))


Resumen por ramo:
                 ramo  polizas  prima_total  pct_cartera
                  GMM    22531 731022322.30        49.93
                 Vida     7510 454332123.32        31.03
                Autos    14752 253823506.30        17.34
Accidentes Personales     5207  24980678.88         1.71
Resumen por agente:
    nombre_agente  polizas  prima_total   comision
      Sofia Perez     1231  37442769.63 3744276.96
      Jose Medina     1227  34800214.16 3480021.42
   Victor Jimenez     1207  34242602.28 3424260.23
  Pablo Dominguez      674  21039960.22 2103996.02
   Gabriela Nunez      662  20690915.49 2069091.55
     Monica Silva      658  20375762.96 2037576.30
    Sandra Garcia      679  20269987.60 2026998.76
     Raul Navarro      645  20155985.20 2015598.52
     Diana Molina      662  20082984.18 2008298.42
       Elena Diaz      668  20065957.85 2006595.78
   Daniel Sanchez      664  19962770.45 1996277.05
 Manuel Dominguez      684  19852287.17 1985228.72
  Fernando Tor

C:\Users\julia\AppData\Local\Temp\ipykernel_31000\279610437.py:35: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_prima = pd.pivot_table(


In [164]:
## Creamos la pivot table de prima por ramo x grupo de edad

pivot_prima = pd.pivot_table(
        cartera_limpia, values='prima_total', index='nombre_largo',
        columns='g_edad', aggfunc='sum', fill_value=0, margins=True, margins_name='TOTAL'
    ).round(0) / 1000  # informacion en miles de pesos

pivot_estado = pd.pivot_table(
        cartera_limpia, values='id_poliza', index='estado',
        columns='ramo', aggfunc='count', fill_value=0, margins=True, margins_name='TOTAL'
    )

print(pivot_prima.to_string())
print("="*65)
print(pivot_estado.to_string())

g_edad                       18-30       31-45       46-60         61+        TOTAL
nombre_largo                                                                       
Accidentes Personales     5248.369    8348.395    8731.668    2652.247    24980.679
Automoviles              55457.022   84225.512   84571.404   29569.569   253823.506
Gastos Medicos Mayores  139278.258  223908.688  264218.298  103617.078   731022.322
Vida Individual          79667.849  134774.641  166301.773   73587.860   454332.123
TOTAL                   279651.498  451257.236  523823.143  209426.754  1464158.631
ramo              Accidentes Personales  Autos    GMM  Vida  TOTAL
estado                                                            
Baja California                     321    990   1508   529   3348
CDMX                                357   1000   1485   515   3357
Chihuahua                           360    972   1501   517   3350
Coahuila                            361    999   1492   514   3366
Estado de 

C:\Users\julia\AppData\Local\Temp\ipykernel_31000\2526670704.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_prima = pd.pivot_table(
C:\Users\julia\AppData\Local\Temp\ipykernel_31000\2526670704.py:8: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_estado = pd.pivot_table(


In [183]:
# Hallazgos clave
ramo_mayor_prima = resumen_ramo.loc[resumen_ramo['prima_total'].idxmax(), 'ramo']
resumen_max_estado = cartera_limpia.groupby('estado', observed=True).agg(
    num_siniestros= ('num_siniestros','sum'),
    num_polizas= ('id_poliza','count'),
    frecuencia_siniestros = ('num_siniestros' , lambda x: x.sum()/x.count() if x.count()>0 else 0)
).reset_index()
 
siniestros_max_estado = resumen_max_estado.loc[
    resumen_max_estado['frecuencia_siniestros'].idxmax(),
    'estado'
]


print(f'Ramo con mayor prima total: {ramo_mayor_prima}')
print(f'Estado con mayor frecuencia de siniestros: {siniestros_max_estado}')

Ramo con mayor prima total: GMM
Estado con mayor frecuencia de siniestros: Puebla


In [184]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE 5: EXPORTAR
# ══════════════════════════════════════════════════════════════════════════════
# Excel con 5 hojas: Cartera_Limpia, Resumen_Ramo, Resumen_Agente,
#                    Pivot_Prima, Pivot_Zona
# Parquet: cartera_q1_2026_final.parquet
# Compara tamano CSV equivalente vs Parquet

# Tu codigo aqui:
with pd.ExcelWriter('datos/reporte_final.xlsx', engine='openpyxl') as writer:
    cartera_limpia.to_excel(writer, sheet_name='Cartera_Limpia', index=False)
    resumen_ramo.to_excel(writer, sheet_name='Resumen_Ramo', index=False)
    resumen_agente.to_excel(writer, sheet_name='Resumen_Agente', index=False)
    pivot_prima.to_excel(writer, sheet_name='Pivot_Prima')
    pivot_estado.to_excel(writer, sheet_name='Pivot_Zona')

kb_excel = os.path.getsize('datos/reporte_final.xlsx')/1024
print(f'Archivo Excel final: {kb_excel:.0f} KB con 5 hojas')

cartera_limpia.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
kb_parquet = os.path.getsize('datos/cartera_q1_2026_final.parquet')/1024
print(f'Archivo Parquet: {kb_parquet:.0f} KB')


Archivo Excel final: 10827 KB con 5 hojas
Archivo Parquet: 2065 KB


In [ ]:
# ── Solucion resumida (descomenta solo si necesitas referencia) ──────────────

# FASE 1:
# df = pd.read_csv('datos/cartera_polizas.csv', usecols=ANALITICAS, na_values=['N/D','N/A'])
# ramos_cat = pd.read_csv('datos/catalogo_ramos.csv')
# agentes_cat = pd.read_csv('datos/catalogo_agentes.csv')

# FASE 2:
# df = df.drop_duplicates()
# df['sexo'] = df['sexo'].str.strip().str.upper().map(MAPA_SEXO)
# for col in ['fecha_emision','fecha_inicio_vigencia','fecha_fin_vigencia']:
#     df[col] = pd.to_datetime(df[col], errors='coerce')
# df['fecha_nacimiento'] = pd.to_datetime(df['fecha_nacimiento'],format='%d/%m/%Y',errors='coerce')
# df['prima_neta'] = df.groupby('ramo')['prima_neta'].transform(lambda x: x.fillna(x.median()))
# df['codigo_postal'] = df['codigo_postal'].replace('N/D', np.nan)
# for col in ['ramo','plan','canal_venta','forma_pago','estado','sexo','tipo_vehiculo']:
#     if col in df.columns: df[col] = df[col].astype('category')

# FASE 3 (extracto):
# df = pd.merge(df, ramos_cat[['ramo','nombre_largo','tasa_base']], on='ramo', how='left')
# df['g_edad'] = pd.cut(df['edad'],bins=[0,30,45,60,100],labels=['18-30','31-45','46-60','61+'])
# df['prima_calc'] = df['suma_asegurada'] * df['tasa_base'] * 1.16
# from mi_modulo import clasificar_riesgo
# df['nivel_riesgo'] = df['prima_calc'].apply(lambda p: 'ALTO' if p>15000 else 'MEDIO' if p>6000 else 'BAJO')

# FASE 5:
# with pd.ExcelWriter('datos/reporte_ejecutivo_Q1_2026.xlsx', engine='openpyxl') as w:
#     df.to_excel(w,'Cartera_Limpia',index=False)
#     resumen_ramo.to_excel(w,'Resumen_Ramo',index=False)
#     resumen_agente.to_excel(w,'Resumen_Agente',index=False)
#     tabla_prima.to_excel(w,'Pivot_Prima')
#     tabla_zona.to_excel(w,'Pivot_Zona')
# df.to_parquet('datos/cartera_q1_2026_final.parquet', index=False)
# print(f'CSV: {df.to_csv(index=False).encode().__len__()/1024:.0f} KB')
# print(f'Parquet: {os.path.getsize("datos/cartera_q1_2026_final.parquet")/1024:.0f} KB')

---
## Resumen: Lo que Aprendiste en la Sesion 8

| Duda | Herramienta | Aprendizaje clave |
|------|-------------|-------------------|
| 46 columnas | `usecols` + taxonomia | Clasificar antes de cargar — decision de negocio |
| Texto sucio | `str.strip/upper/map` | Normalizar ANTES del primer groupby |
| Fechas texto | `pd.to_datetime(errors='coerce')` | Nunca detener el pipeline por fechas invalidas |
| JSON anidado | `json_normalize(sep='_')` | Aplanar antes de analizar |
| 90k filas | `chunksize` + `category` | Medir memoria antes de decidir |
| Downcast riesgoso | Regla float32 < 100k MXN | float64 para montos grandes siempre |
| Polars | `pl.scan_csv().collect()` | Lazy evaluation = optimizacion automatica |

**T5 Pandas — COMPLETADO**

**Proxima sesion — Mie 6 mayo, 18:00 h:**
T6 Visualizacion — Matplotlib, Seaborn y Plotly.

```bash
git add sesion8_M1_notebook.ipynb
git commit -m "Sesion 8: pipeline completo datos reales - str datetime JSON Polars"
git push
```

---
*Diplomado ML en Seguros · FC UNAM · 2026*